In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1800016313791275
epoch:  1, loss: 0.17747145891189575
epoch:  2, loss: 0.1750938445329666
epoch:  3, loss: 0.17204242944717407
epoch:  4, loss: 0.1686505824327469
epoch:  5, loss: 0.1655101180076599
epoch:  6, loss: 0.16252312064170837
epoch:  7, loss: 0.15953725576400757
epoch:  8, loss: 0.15667927265167236
epoch:  9, loss: 0.15388235449790955
epoch:  10, loss: 0.1511380970478058
epoch:  11, loss: 0.14846265316009521
epoch:  12, loss: 0.14583683013916016
epoch:  13, loss: 0.14327023923397064
epoch:  14, loss: 0.14076487720012665
epoch:  15, loss: 0.13831695914268494
epoch:  16, loss: 0.1359434276819229
epoch:  17, loss: 0.1336163729429245
epoch:  18, loss: 0.13134737312793732
epoch:  19, loss: 0.12912079691886902
epoch:  20, loss: 0.12694577872753143
epoch:  21, loss: 0.12480688095092773
epoch:  22, loss: 0.1227131336927414
epoch:  23, loss: 0.12067218869924545
epoch:  24, loss: 0.11865338683128357
epoch:  25, loss: 0.11667603999376297
epoch:  26, loss: 0.1147399619

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.26489508152008057
Test metrics:  R2 = -0.12936735153198242


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2242005169391632
epoch:  1, loss: 1730108719104.0
epoch:  2, loss: 4.178115719290319e+32
epoch:  3

_LinAlgError: torch.linalg.solve: The solver failed because the input matrix is singular.

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9931288957595825
Test metrics:  R2 = 0.9775075316429138
